In [1]:
import pandas as pd
import numpy as np

In [4]:
# Set random seed for reproducibility
np.random.seed(42)
# Number of simulated users
n = 50000

In [13]:
# 1. CREATE RAW BASE DATASET

df = pd.DataFrame({
    "user_id": range(1, n + 1),

    # Random assignment to pricing groups
    "test_group": np.random.choice(["A_basic", "B_premium"], size=n),

    # Demographics
    "age": np.random.randint(18, 70, size=n),

    # Income segmentation
    "income_level": np.random.choice(
        ["low", "medium", "high"],
        size=n,
        p=[0.4, 0.4, 0.2]
    ),

    # Risk score from 0 to 1
    "risk_score": np.random.beta(2, 5, size=n),

    # Whether user had a previous insurance claim
    "previous_claim": np.random.choice([0, 1], size=n, p=[0.8, 0.2]),

    # Acquisition channel
    "channel": np.random.choice(["online", "agent"], size=n, p=[0.6, 0.4]),

    # Swedish city segmentation with intentional inconsistencies
    "city": np.random.choice(
        [
            "Stockholm", "Gothenburg", "Malmo", "Uppsala", "Other",
            "stockholm", "STOCKHOLM", "Gbg", "malmo "
        ],
        size=n,
        p=[0.25, 0.17, 0.13, 0.09, 0.22, 0.04, 0.03, 0.04, 0.03]
    )
})

# Fix dtypes to avoid assignment errors later
df["age"] = df["age"].astype("int64")
df["risk_score"] = df["risk_score"].astype("float64")

In [14]:
# 2. PRICING DESIGN
# Assign monthly insurance price based on test group
df["price"] = np.where(df["test_group"] == "A_basic", 279, 479)

df["price_currency"] = "SEK"
df["billing_period"] = "monthly"


In [15]:
# 3. CONVERSION PROBABILITY MODEL
# Lower price has higher baseline conversion
base_conversion = np.where(df["test_group"] == "A_basic", 0.12, 0.09)

# Higher risk users are more likely to buy insurance
risk_effect = df["risk_score"] * 0.15

# Users with previous claims are more likely to buy insurance
claim_effect = df["previous_claim"] * 0.10

# Income affects ability or willingness to buy insurance
income_effect = df["income_level"].map({
    "low": -0.03,
    "medium": 0.0,
    "high": 0.04
})

In [16]:
# Agent-assisted channel converts better than online
channel_effect = df["channel"].map({
    "online": -0.01,
    "agent": 0.02
})

In [17]:
# City-level conversion effect
city_effect = df["city"].str.strip().str.lower().map({
    "stockholm": 0.02,
    "gothenburg": 0.01,
    "gbg": 0.01,
    "malmo": 0.005,
    "uppsala": 0.0,
    "other": -0.01
})

In [18]:
# Combine effects into final conversion probability
conversion_prob = (
    base_conversion
    + risk_effect
    + claim_effect
    + income_effect
    + channel_effect
    + city_effect
)

# Keep probability within realistic range
conversion_prob = conversion_prob.clip(0.01, 0.6)

# Simulate conversion outcome
df["converted"] = np.random.binomial(1, conversion_prob)

# Revenue is generated only when user converts
df["revenue"] = df["converted"] * df["price"]

In [21]:
# 4. ADD RAW DATA QUALITY ISSUES

# Add inconsistent channel formatting
dirty_channel_idx = np.random.choice(df.index, size=int(n * 0.03), replace=False)
df.loc[dirty_channel_idx, "channel"] = df.loc[dirty_channel_idx, "channel"].apply(
    lambda x: f" {x.upper()} "
)

# Add missing income values
missing_income_idx = np.random.choice(df.index, size=int(n * 0.01), replace=False)
df.loc[missing_income_idx, "income_level"] = np.nan

# Add missing city values
missing_city_idx = np.random.choice(df.index, size=int(n * 0.005), replace=False)
df.loc[missing_city_idx, "city"] = np.nan

# Add unrealistic ages
age_issue_idx = np.random.choice(df.index, size=int(n * 0.002), replace=False)
df.loc[age_issue_idx, "age"] = np.random.choice([-30, 125], size=len(age_issue_idx))

# Add invalid risk scores
risk_issue_idx = np.random.choice(df.index, size=int(n * 0.002), replace=False)
df.loc[risk_issue_idx, "risk_score"] = np.random.choice([-0.2, 1.2], size=len(risk_issue_idx))

# Add duplicate rows
duplicate_rows = df.sample(n=int(n * 0.005), random_state=42)
df = pd.concat([df, duplicate_rows], ignore_index=True)

In [22]:
# 5. SAVE RAW DATASET

df.to_csv("insurance_pricing_ab_test_raw.csv", index=False)

# Quick raw data check
print(df.head())
print(df.shape)
print(df.isna().sum())
print(df.duplicated().sum())

   user_id test_group  age income_level  risk_score  previous_claim channel  \
0        1  B_premium   57       medium    0.328576               1   agent   
1        2  B_premium   45       medium    0.253105               0   agent   
2        3  B_premium   51          low    0.272809               0   agent   
3        4  B_premium   35          low    0.314791               0   agent   
4        5    A_basic   39          low    0.074665               0  online   

         city  price price_currency billing_period  converted  revenue  
0       Other    479            SEK        monthly          1      479  
1  Gothenburg    479            SEK        monthly          0        0  
2   Stockholm    479            SEK        monthly          0        0  
3         Gbg    479            SEK        monthly          0        0  
4  Gothenburg    279            SEK        monthly          0        0  
(50500, 13)
user_id             0
test_group          0
age                 0
income_le

In [23]:
print(df.head())

   user_id test_group  age income_level  risk_score  previous_claim channel  \
0        1  B_premium   57       medium    0.328576               1   agent   
1        2  B_premium   45       medium    0.253105               0   agent   
2        3  B_premium   51          low    0.272809               0   agent   
3        4  B_premium   35          low    0.314791               0   agent   
4        5    A_basic   39          low    0.074665               0  online   

         city  price price_currency billing_period  converted  revenue  
0       Other    479            SEK        monthly          1      479  
1  Gothenburg    479            SEK        monthly          0        0  
2   Stockholm    479            SEK        monthly          0        0  
3         Gbg    479            SEK        monthly          0        0  
4  Gothenburg    279            SEK        monthly          0        0  


In [24]:
print(df.shape)

(50500, 13)


In [25]:
print(df.isna().sum())

user_id             0
test_group          0
age                 0
income_level      997
risk_score          0
previous_claim      0
channel             0
city              504
price               0
price_currency      0
billing_period      0
converted           0
revenue             0
dtype: int64


In [26]:
print(df.duplicated().sum())

481


In [27]:
print(df.duplicated(subset="user_id").sum())



500
